In [40]:
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
import faiss

In [ ]:
# The SIFT dataset vectors are stored in .fvecs format. 
# Each vector entry starts with an int32 dimension prefix followed by that many float32 components.

def read_ivecs(fname: str) -> np.ndarray:
    with open(fname, "rb") as f:
        data = np.fromfile(f, dtype=np.int32)

    dim = data[0]
    data = data.reshape(-1, dim + 1)

    return data[:, 1:]

def read_fvecs(fname: str) -> np.ndarray:
    with open(fname, "rb") as f:
        data = np.fromfile(f, dtype=np.int32)

    dim = data[0]
    data = data.reshape(-1, dim + 1)

    # reinterpret everything except first column as float32
    vectors = data[:, 1:].view(np.float32)

    return vectors

BASE = "/home/adamm/Desktop/thesis/siftsmall/siftsmall_base.fvecs"
QUERY = "/home/adamm/Desktop/thesis/siftsmall/siftsmall_query.fvecs"
TRUTH = "/home/adamm/Desktop/thesis/siftsmall/siftsmall_groundtruth.ivecs"

base_vectors = read_fvecs(BASE)
query_vectors = read_fvecs(QUERY)
truth_vectors = read_ivecs(TRUTH)

print(f"Dimensions of base: {base_vectors.shape}    Type: {base_vectors.dtype}")
print(f"Dimensions of query: {query_vectors.shape}    Type: {query_vectors.dtype}")
print(f"Dimensions of truth: {truth_vectors.shape}    Type: {truth_vectors.dtype}")

Dimensions of base: (10000, 128)    Type: float32
Dimensions of query: (100, 128)    Type: float32
Dimensions of truth: (100, 100)    Type: int32


In [25]:
n, d = base_vectors.shape
ids = np.arange(n)

In [59]:
# Medium correlation
def add_noise(attr, noise_fraction, k):
    n = len(attr)
    noisy = attr.copy()
    noise_idx = np.random.choice(n, int(noise_fraction * n), replace=False)
    noisy[noise_idx] = np.randint(0, k, size=len(noise_idx))
    return noisy
                                

In [ ]:
# Uncorrelated
def generate_uncorrelated(n, k, seed=42):
    np.random.seed(seed)
    return np.random.randint(0, k, size=n)

# Selectivity for equality predicate ~ 1/k 

# LCPS: Low Cardinality Predicate Space
attr_lc_uncorrelated = generate_uncorrelated(n, k=5)

# HCPS: High Cardinality Predicate Space
attr_hc_uncorrelated = generate_uncorrelated(n, k=500)

In [ ]:
# Correlated
def generate_correlated(vectors, k, seed=42):
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        random_state=seed,
        batch_size=1024,
        max_iter=100
    )
    return kmeans.fit_predict(vectors)

# attr_lc = cluster_id
# LCPS: Low Cardinality Predicate Space
attr_lc_correlated = generate_correlated(base_vectors, k=5)

# HCPS: High Cardinality Predicate Space
attr_hc_correlated = generate_correlated(base_vectors, k=500)

In [44]:
index_flat = faiss.IndexFlatL2(d)
index_flat.add(base_vectors)

print(f"Total vectors in index: {index_flat.ntotal}")

distances, indices = index_flat.search(query_vectors, 10)

Total vectors in index: 10000


In [57]:
def recall_at_k(pred, gt, k):
    correct = 0
    total = pred.shape[0]
    
    for i in range(total):
        top_k_results = pred[i][:k] 
        ground_truth_k_neighbors = gt[i][:k]
        correct += len(set(top_k_results) & set(ground_truth_k_neighbors)) > 0
    return correct / total

print(f'Recall@10: {recall_at_k(indices, truth_vectors, 10)}')

Recall@10: 1.0
